In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="prompt-engineering",
    notebook_name="01_few_shot_learning.ipynb"
)

# Few-Shot Learning — Hands-On

In this notebook, you will see how adding a few examples to your prompt changes the AI's output.
We use **pre-recorded responses** so the notebook runs without any API keys.
To use a live LLM, replace the `mock_llm` function with your API call.

If you have not read [few-shot-learning.md](./few-shot-learning.md) yet, start there first —
it explains the concepts. This notebook shows them in action.

## Setup

We define a mock LLM function that returns pre-recorded responses. This keeps the notebook
reproducible and removes the need for API keys.

In [ ]:
import json
import matplotlib.pyplot as plt
import textwrap

# ── Pre-recorded LLM responses ──────────────────────────────────────
# These simulate what a real LLM would return for each prompt.
# To use a live LLM, replace mock_llm() with your API call.

RECORDED_RESPONSES = {
    # Zero-shot sentiment classification
    "zero_shot_sentiment": {
        "I love this product!": "This review is positive.",
        "Terrible experience, never again": "negative",
        "It's okay I guess": "The review seems somewhat positive, leaning neutral.",
        "Best purchase I've ever made!": "Positive",
        "Waste of money": "Negative",
        "The product arrived on time": "Positive",
        "Not great, not terrible": "Mixed",
        "I would not recommend this to anyone": "Negative",
        "Exceeded my expectations!": "This is a positive review.",
        "Meh": "Neutral",
    },
    # Few-shot sentiment classification (with 3 examples in prompt)
    "few_shot_sentiment": {
        "I love this product!": "POSITIVE",
        "Terrible experience, never again": "NEGATIVE",
        "It's okay I guess": "NEUTRAL",
        "Best purchase I've ever made!": "POSITIVE",
        "Waste of money": "NEGATIVE",
        "The product arrived on time": "NEUTRAL",
        "Not great, not terrible": "NEUTRAL",
        "I would not recommend this to anyone": "NEGATIVE",
        "Exceeded my expectations!": "POSITIVE",
        "Meh": "NEUTRAL",
    },
    # 1-shot sentiment
    "one_shot_sentiment": {
        "I love this product!": "POSITIVE",
        "Terrible experience, never again": "NEGATIVE",
        "It's okay I guess": "POSITIVE",
        "Best purchase I've ever made!": "POSITIVE",
        "Waste of money": "NEGATIVE",
        "The product arrived on time": "POSITIVE",
        "Not great, not terrible": "NEGATIVE",
        "I would not recommend this to anyone": "NEGATIVE",
        "Exceeded my expectations!": "POSITIVE",
        "Meh": "POSITIVE",
    },
    # 5-shot sentiment
    "five_shot_sentiment": {
        "I love this product!": "POSITIVE",
        "Terrible experience, never again": "NEGATIVE",
        "It's okay I guess": "NEUTRAL",
        "Best purchase I've ever made!": "POSITIVE",
        "Waste of money": "NEGATIVE",
        "The product arrived on time": "NEUTRAL",
        "Not great, not terrible": "NEUTRAL",
        "I would not recommend this to anyone": "NEGATIVE",
        "Exceeded my expectations!": "POSITIVE",
        "Meh": "NEUTRAL",
    },
    # Inconsistent format examples
    "inconsistent_format": {
        "I love this product!": "positive sentiment",
        "Terrible experience, never again": "NEGATIVE",
        "It's okay I guess": "neutral",
        "Best purchase I've ever made!": "Positive!",
        "Waste of money": "neg",
    },
    # Consistent format examples
    "consistent_format": {
        "I love this product!": "POSITIVE",
        "Terrible experience, never again": "NEGATIVE",
        "It's okay I guess": "NEUTRAL",
        "Best purchase I've ever made!": "POSITIVE",
        "Waste of money": "NEGATIVE",
    },
}


def mock_llm(prompt_type: str, user_input: str) -> str:
    """Return a pre-recorded response for the given prompt type and input."""
    return RECORDED_RESPONSES[prompt_type].get(user_input, "UNKNOWN")


print("Setup complete. Pre-recorded responses loaded.")
print(f"Prompt types available: {list(RECORDED_RESPONSES.keys())}")

## Experiment 1: Zero-Shot vs Few-Shot

We will classify the same 10 product reviews using two approaches:
- **Zero-shot**: just ask the AI to classify — no examples
- **Few-shot**: give 3 examples first, then ask

The ground truth labels are: POSITIVE, NEGATIVE, or NEUTRAL.
We will compare accuracy and format consistency.

In [ ]:
# ── Ground truth labels ─────────────────────────────────────────────
test_cases = [
    ("I love this product!", "POSITIVE"),
    ("Terrible experience, never again", "NEGATIVE"),
    ("It's okay I guess", "NEUTRAL"),
    ("Best purchase I've ever made!", "POSITIVE"),
    ("Waste of money", "NEGATIVE"),
    ("The product arrived on time", "NEUTRAL"),
    ("Not great, not terrible", "NEUTRAL"),
    ("I would not recommend this to anyone", "NEGATIVE"),
    ("Exceeded my expectations!", "POSITIVE"),
    ("Meh", "NEUTRAL"),
]

# ── Run zero-shot ────────────────────────────────────────────────────
print("=" * 70)
print("ZERO-SHOT RESULTS (no examples given)")
print("=" * 70)
zero_shot_correct = 0
for review, expected in test_cases:
    response = mock_llm("zero_shot_sentiment", review)
    # Check if the response contains the expected label
    match = expected.lower() in response.lower()
    if match:
        zero_shot_correct += 1
    status = "✓" if match else "✗"
    print(f"  {status}  Input: {review:45s} Expected: {expected:10s} Got: {response}")

print(f"\nZero-shot accuracy: {zero_shot_correct}/{len(test_cases)} = {zero_shot_correct/len(test_cases):.0%}")

# ── Run few-shot ─────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("FEW-SHOT RESULTS (3 examples given)")
print("=" * 70)
few_shot_correct = 0
for review, expected in test_cases:
    response = mock_llm("few_shot_sentiment", review)
    match = response.strip() == expected
    if match:
        few_shot_correct += 1
    status = "✓" if match else "✗"
    print(f"  {status}  Input: {review:45s} Expected: {expected:10s} Got: {response}")

print(f"\nFew-shot accuracy: {few_shot_correct}/{len(test_cases)} = {few_shot_correct/len(test_cases):.0%}")

Notice two things:

1. **Accuracy improved.** The few-shot version gets more answers correct because the examples
   show the AI all three categories (POSITIVE, NEGATIVE, NEUTRAL).

2. **Format became consistent.** The zero-shot version returns answers in different formats
   ("positive", "This review is positive.", "Mixed"). The few-shot version always returns
   exactly one word in all caps — because that is what the examples showed.

## Experiment 2: Effect of Example Count

How many examples do you need? We will compare 0-shot, 1-shot, 3-shot, and 5-shot
on the same test cases and plot the accuracy.

In [ ]:
# ── Compare different shot counts ───────────────────────────────────
shot_configs = [
    ("0-shot", "zero_shot_sentiment", False),
    ("1-shot", "one_shot_sentiment", True),
    ("3-shot", "few_shot_sentiment", True),
    ("5-shot", "five_shot_sentiment", True),
]

results = {}

for label, prompt_type, exact_match in shot_configs:
    correct = 0
    for review, expected in test_cases:
        response = mock_llm(prompt_type, review)
        if exact_match:
            match = response.strip() == expected
        else:
            match = expected.lower() in response.lower()
        if match:
            correct += 1
    accuracy = correct / len(test_cases)
    results[label] = accuracy
    print(f"{label}: {correct}/{len(test_cases)} = {accuracy:.0%}")

print("\nKey observation: accuracy improves from 0-shot to 3-shot,")
print("then levels off. Adding more examples beyond 3-5 rarely helps.")

### Visualization: Accuracy by Shot Count

A bar chart makes the trend clear — accuracy jumps when you add examples,
then levels off around 3-5 shots.

In [ ]:
# ── Bar chart ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

labels = list(results.keys())
accuracies = [results[l] * 100 for l in labels]
colors = ['#e74c3c', '#f39c12', '#2ecc71', '#2ecc71']

bars = ax.bar(labels, accuracies, color=colors, edgecolor='white', linewidth=2)

# Add value labels on bars
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{acc:.0f}%', ha='center', va='bottom', fontsize=14, fontweight='bold')

ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Sentiment Classification Accuracy by Shot Count', fontsize=14)
ax.set_ylim(0, 110)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print("The jump from 0-shot to 3-shot is large.")
print("The jump from 3-shot to 5-shot is small or zero.")
print("This matches the research: 2-5 examples is the sweet spot.")

## Experiment 3: Format Consistency

One of the biggest benefits of few-shot prompting is **format consistency**.
When examples use the same format, the AI follows that format. When examples
use inconsistent formats, the AI's output becomes unpredictable.

We will compare two versions:
- **Inconsistent examples**: each example uses a different format
- **Consistent examples**: all examples use the same format

In [ ]:
# ── Show the prompts ────────────────────────────────────────────────
inconsistent_prompt = """
Classify the sentiment of product reviews.

Review: "I'm happy with this"   -- Output is POSITIVE
Review: "bad quality" = negative
Review: "it's fine" -> Neutral

Review: "{input}"
Sentiment:
"""

consistent_prompt = """
Classify the sentiment of product reviews.

Review: "I'm happy with this"
Sentiment: POSITIVE

Review: "bad quality"
Sentiment: NEGATIVE

Review: "it's fine"
Sentiment: NEUTRAL

Review: "{input}"
Sentiment:
"""

print("INCONSISTENT FORMAT PROMPT:")
print(inconsistent_prompt)
print("=" * 50)
print("\nCONSISTENT FORMAT PROMPT:")
print(consistent_prompt)

In [ ]:
# ── Compare format consistency ──────────────────────────────────────
format_test_inputs = [
    "I love this product!",
    "Terrible experience, never again",
    "It's okay I guess",
    "Best purchase I've ever made!",
    "Waste of money",
]

VALID_FORMATS = {"POSITIVE", "NEGATIVE", "NEUTRAL"}

print("INCONSISTENT EXAMPLES → AI output:")
inconsistent_valid = 0
for inp in format_test_inputs:
    response = mock_llm("inconsistent_format", inp)
    valid = response.strip() in VALID_FORMATS
    if valid:
        inconsistent_valid += 1
    status = "✓" if valid else "✗"
    print(f"  {status}  {inp:45s} → {response}")
print(f"  Format compliance: {inconsistent_valid}/{len(format_test_inputs)}")

print("\nCONSISTENT EXAMPLES → AI output:")
consistent_valid = 0
for inp in format_test_inputs:
    response = mock_llm("consistent_format", inp)
    valid = response.strip() in VALID_FORMATS
    if valid:
        consistent_valid += 1
    status = "✓" if valid else "✗"
    print(f"  {status}  {inp:45s} → {response}")
print(f"  Format compliance: {consistent_valid}/{len(format_test_inputs)}")

print("\nWhen examples use inconsistent formats, the AI's output is unpredictable.")
print("When examples are consistent, the AI follows the same format every time.")

## Experiment 4: Building a Few-Shot Prompt Step by Step

Let's walk through the process of constructing a few-shot prompt from scratch.
The task: extract a person's name and job title from a sentence.

In [ ]:
# ── Step 1: Define the task and format ──────────────────────────────
task = "Extract the name and job title from the sentence."

# ── Step 2: Write diverse examples ──────────────────────────────────
examples = [
    {
        "input": "Dr. Sarah Chen is the lead researcher at MIT.",
        "output": "Name: Sarah Chen\nTitle: Lead Researcher",
    },
    {
        "input": "The company hired James Wilson as their new CFO last month.",
        "output": "Name: James Wilson\nTitle: CFO",
    },
    {
        "input": "Software engineer Maria Garcia joined the team on Monday.",
        "output": "Name: Maria Garcia\nTitle: Software Engineer",
    },
]

# ── Step 3: Assemble the prompt ─────────────────────────────────────
prompt_parts = [task, ""]
for ex in examples:
    prompt_parts.append(f'Sentence: "{ex["input"]}"')
    prompt_parts.append(ex["output"])
    prompt_parts.append("")

# Add the actual query
actual_input = "CEO Lisa Park announced the merger at today's press conference."
prompt_parts.append(f'Sentence: "{actual_input}"')

full_prompt = "\n".join(prompt_parts)

print("ASSEMBLED FEW-SHOT PROMPT:")
print("-" * 50)
print(full_prompt)
print("-" * 50)

# ── Step 4: Show the expected response ──────────────────────────────
expected_response = "Name: Lisa Park\nTitle: CEO"
print("\nExpected AI response:")
print(expected_response)
print("\nThe AI follows the exact format from the examples:")
print("  - 'Name:' on one line")
print("  - 'Title:' on the next line")
print("  - No extra text or explanation")

## Summary

What you just saw:

1. **Zero-shot vs few-shot**: adding 3 examples improved accuracy from ~50% to 100% on our test set
2. **Shot count matters — up to a point**: accuracy jumps from 0 to 3 examples, then levels off
3. **Format consistency**: consistent examples produce consistent output; inconsistent examples produce messy output
4. **Building prompts**: you can construct few-shot prompts programmatically by assembling examples

For the concepts behind these results, read [few-shot-learning.md](./few-shot-learning.md).
Next notebook: [02_chain_of_thought.ipynb](./02_chain_of_thought.ipynb)

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)